# PC-FNO3D NATIONAL v4 — C' s44 + OI des innovations des aéroports (VALIDATION 2021)

Règle `configs/national/regle_oi_post_v4.json` (sha256 `a1fa8921faae…`), vérifiée par le carnet. Post-traitement sans réentraînement : vent à 10 m = modèle + interpolation optimale des écarts (mesure − modèle) des **autres** aéroports (L 150 km, λ 4, fixés sur 2015–2020). Chaque aéroport est jugé retiré. **Aucune lecture de 2022, aucun entraînement, aucun bruit ajouté.**


## 0. GPU

In [ ]:
# Le national (453 M parametres, grille 16 x 828 x 388) ne tient pas sur une L4 de 22 Go, meme avec le
# recalcul des activations (3 echecs le 11/11, dont un avec le carnet corrige : 20,8 Go + 2,45 Go demandes).
# Si Colab n'attribue pas un GPU d'au moins 30 Go (A100), la session est liberee AVANT le montage du Drive.
import subprocess as _sp
_g = _sp.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
             capture_output=True, text=True)
_nom, _mo = ([s.strip() for s in _g.stdout.strip().splitlines()[0].split(",")] if _g.returncode == 0
             and _g.stdout.strip() else ("aucun GPU", "0"))
print("GPU attribue :", _nom, f"{float(_mo) / 1024:.0f} Go")
if float(_mo) < 30 * 1024:
    from google.colab import runtime
    print("GPU trop petit pour le national : session liberee, rien n'a ete lance. Relancer plus tard (A100).")
    runtime.unassign()
    raise SystemExit("GPU de moins de 30 Go")

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "19341d0d676872f77dec780765afa80fe33868a8"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_oi_post_v4.json", "a1fa8921faae16dabdaf4f6c5e88599354cc66097bb2e71bb311e2cac7ad146c"), ("regle_choix_v4_Cp_s44.json", "19cee4e50461e75a74f1ad54db7c4fb9496c3f5be112e65bcbb43278862ce4a5"), ("regle_w_servi_v4.json", "4e5453455d67c2e0b397adbe88852d47f898c6827fa250a933f588579460e5b3"), ("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"),):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regle v4 verifiee")

## 2. Cache v3

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
# controle de la verite deja fait par les carnets d'entrainement

## 3. Mémoire du GPU

In [ ]:
# Essai de memoire dans un PROCESSUS SEPARE : le noyau du carnet ne touche jamais au GPU (le 11/11, l'essai fait
# dans le noyau y gardait 3,3 Go et l'entrainement sur L4 22 Go a manque de memoire). Allocateur a segments
# extensibles (moins de fragmentation), herite par les commandes lancees ensuite. Aucun effet sur les calculs.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ESSAI = r"""
import json, sys, torch
from pcfno3d.config import Config
from pcfno3d.geometry import Geometry
from pcfno3d.model import build_model
recalcul = sys.argv[1] == "1"
cfg = Config.load("configs/national_1km_colab.yaml")
cfg.model.recalcul_activations = recalcul
geom = Geometry(cfg, device="cuda")
m = build_model(cfg, geom.c_in).cuda().train()
x = torch.randn(cfg.train.batch_size, geom.c_in, *geom.shape, device="cuda")
torch.cuda.reset_peak_memory_stats()
try:
    with torch.autocast("cuda", dtype=torch.bfloat16):
        m(x).float().pow(2).mean().backward()
    pic = torch.cuda.max_memory_allocated() / 1e9
except torch.cuda.OutOfMemoryError:
    pic = None
p = torch.cuda.get_device_properties(0)
print("ESSAI", json.dumps({"gpu": p.name, "total": p.total_memory / 1e9, "pic": pic}))
"""
Path("/content/essai_memoire.py").write_text(ESSAI)
def essai(recalcul: bool) -> dict:
    r = subprocess.run([sys.executable, "/content/essai_memoire.py", "1" if recalcul else "0"], cwd="/content/PFE",
                       capture_output=True, text=True)
    ligne = next((l for l in r.stdout.splitlines() if l.startswith("ESSAI ")), None)
    if ligne is None:
        print(r.stdout[-2000:], r.stderr[-2000:])
        return {"gpu": "?", "total": float("nan"), "pic": None}
    return json.loads(ligne[6:])
e = essai(False)
total = e["total"]
RECALCUL = e["pic"] is None or e["pic"] > 0.85 * total
if RECALCUL:
    e = essai(True)
print(e["gpu"], f"{total:.0f} Go | pic de l'essai {e['pic'] if e['pic'] is None else round(e['pic'], 1)} Go "
      f"| recalcul des activations : {RECALCUL}")
if total < 30:
    print("ATTENTION : GPU de moins de 30 Go (L4 ?). Le national a manque de memoire sur L4 le 11/11 avec l'ancien "
          "carnet ; cet essai ne compte ni l'optimiseur ni l'EMA. Preferer une A100.")
OPTS = f"--set model.recalcul_activations={str(RECALCUL).lower()}"

## 4. Aéroports (validation, aéroport retiré), puis OI ; fin de la session

In [ ]:
import shlex, yaml, traceback
from google.colab import runtime
RUN = NAT / "runs" / "national_1km_v4_Cp_s44"          # entrainement fixe par regle_choix_v4_Cp_s44
c = yaml.safe_load(open(CODE / "configs" / "national_1km_colab.yaml"))
c["name"] = "national_v4_eval"
c["paths"]["run_dir"], c["paths"]["local_dir"] = str(RUN), "/content/run_local_v4_sCp_s44"
yaml.safe_dump(c, open(CODE / "configs" / "national_v4_eval.yaml", "w"), sort_keys=False)
try:
    # VALIDATION 2021 seulement : chaque aeroport retire de l'entree quand il est juge (aucune lecture de 2022)
    if not (RUN / "validation_reelle" / "val" / "stations_paires.csv").exists():
        lancer("validation_reelle_v4_validation", f"python -u scripts/valider_grand_tunis_reel.py -c national_v4_eval "
                                                  f"--obs {G / 'obs'} --jeu val")
    lancer("oi_post_v4_validation", f"python -u scripts/oi_post_aeroports.py --run {shlex.quote(str(RUN))} --jeu val")
    print((RUN / "validation_reelle" / "val" / "oi_post" / "tableau.txt").read_text())
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()
